# Social UMAP
This notebook contains the code to generate the social UMAPs from Nair et al. 2025, "Sex-specific behavioral feedback modulates sensorimotor processing and drives flexible social behavior" (bioRxiv, May 2025).


In [ ]:
# imports and configurations
import os
import pandas as pd
import numpy as np
from scipy import signal
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from embedding_utils import (get_freq_scales, compute_wavelet,
                             umap_embedding,
                             time_delay_embedding,
                             SpatialClustering,
                             umap_multiple_fits,
                             basis_transformation,
                             kernel_density_estimation)
from glm_utils import bases
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from scipy.stats import mannwhitneyu, binned_statistic_2d

plt.style.use('ncb.mplstyle')

colors = [mcolors.CSS4_COLORS["violet"], mcolors.CSS4_COLORS["royalblue"],
          mcolors.CSS4_COLORS["thistle"], mcolors.CSS4_COLORS["lightsteelblue"]]
color_palette = sns.set_palette(sns.color_palette(colors))


## How to arrange the data and specify parameters

The data should be stored in the `dat` folder as one CSV file per trial. The columns of each CSV file are used as input features for the UMAP.

The `trial_names` indicate the trials belonging to different groups, namely male-female and male-male interactions. The `features` list indicates the input features used to fit the UMAP, and `condition` indicates the condition on which the UMAPs are compared.

- `start_frame_idx`: index of the first frame of each trial to be used in the analysis

- `end_frame_idx`: index of the last frame of each trial to be used in the analysis

- `confidence_threshold`: confidence level of tracked poses below which frames are ignored in the analysis

### Lowpass filtering
- `required_sampling_rate`: sampling rate of the data

- `filter_features`: flag indicating whether to lowpass filter UMAP input features

- `filter_cutoff`: cutoff frequency if lowpass filtering is used

- `filter_ord`: filter order

### Normalization of input features
- `normalize_features`: whether to normalize the input features

- `normalization`: what type of normalization is used, "standardize", "minmax", or `None`

- `features_range`: for minmax normalization, the feature range to be used as minimum and maximum values. If used, this should be a dictionary whose keys are any or some of the feature names (`features`) and values are a list of two elements: the minimum and maximum value for the feature

### Time delay embedding parameters
- `time_delay_poses`: whether to use time-delay embedding of input features

- `nb_delays`: if `time_delay_poses` is `True`, specify the number of delays to embed

- `basis_transform`: whether to transform the basis vectors of time-delay embedded features

- `basis_function`: if `basis_transform` is `True`, the function to be used for basis transformation

### Wavelet transform parameters
- `wavelet_transform`: whether to wavelet transform the input features. Either time-delay embedding or wavelet transform is used

- `min_freq`: minimum frequency used for wavelet transform

- `max_freq`: maximum frequency used for wavelet transform

- `nfreqs`: number of frequency components to be used for wavelet transform

- `freq_spacing`: frequency spacing to be used: 'dyadic', 'log', or 'uniform'

### UMAP parameters

- `train_size`: ratio of data to be used for fitting the UMAP

- `min_dist`: UMAP hyperparameter `min_dist` (https://umap-learn.readthedocs.io/en/latest/parameters.html)

- `n_neighbors`: UMAP hyperparameter `n_neighbors` (https://umap-learn.readthedocs.io/en/latest/parameters.html)

- `random_state`: random seed used to fit UMAP reproducibly

- `nan_handling`: whether to 'remove' or 'interpolate' NaN values before analysis

### kernel density estimation (KDE) and watershed segmentation parameters
- `bw`: bandwidth to be used for KDE (https://kdepy.readthedocs.io/en/latest/introduction.html#Selecting-a-suitable-bandwidth)

- `nb_gridpoints`: number of grid points on each 2D axis to be used for KDE

- `watershed_threshold`: threshold to be used for watershed segmentation algorithm (see `watershed_segmentation` in `embedding_utils.py`)

- `results_save_path`: path to save social UMAP results

The notebook reads data from the CSV files and stores them in `preprocessed_data_path`.


In [ ]:
data_save_path = "dat"

preprocessed_data_path = os.path.join(data_save_path, "preprocessed_data.npz")

# parameters to get data
trial_names = {}
trial_names['male-female'] = \
    ['localhost-20200706_122314',
    'localhost-20200708_122011',
    'localhost-20200707_113229',
    'localhost-20200710_120433',
    'localhost-20200710_134643',
    'localhost-20200709_123809',
    'localhost-20200709_115547',
    'localhost-20200710_131755',
    'localhost-20220214_114900',
    'localhost-20220214_122614',
    'localhost-20220214_130838',
    'localhost-20220214_132737',
    'localhost-20220216_093349',
    'localhost-20220216_101942',
    'localhost-20220218_114657',
    'localhost-20220224_120331',
    'localhost-20220225_121823',
    'localhost-20220302_113245',
    'localhost-20220302_125321',
    'localhost-20220502_115227',
    'localhost-20210127_145608',
    'localhost-20210210_094512',
    'localhost-20210210_100441',
    'localhost-20210617_121041',
    'localhost-20210617_123045',
    'localhost-20220502_123531',
    'localhost-20220506_130520',
    'localhost-20220509_115048',
    'localhost-20220509_124215',
    'localhost-20220511_115515',
     ]

trial_names['male-male'] = \
    ['localhost-20200706_132556',
    'localhost-20200708_120051',
    'localhost-20200812_143257',
    'localhost-20200709_111319',
    'localhost-20200812_155855',
    'localhost-20210222_144954',
    'localhost-20210222_152829',
    'localhost-20210308_120623',
    'localhost-20210308_122507',
    'localhost-20220214_120801',
    'localhost-20220214_135209',
    'localhost-20220222_122109',
    'localhost-20220223_121135',
    'localhost-20220224_124055',
    'localhost-20220224_130031',
    'localhost-20220225_123838',
    'localhost-20220225_125855',
    'localhost-20220302_115211',
    'localhost-20220302_131417',
    'localhost-20220302_133433',
    'localhost-20220502_125611',
    'localhost-20210201_124220',
    'localhost-20210201_132121',
    'localhost-20210209_114709',
    'localhost-20210209_122417',
    'localhost-20220304_121729',
    'localhost-20220304_123618',
    'localhost-20220502_133927',
    'localhost-20220509_121352',
    'localhost-20220509_130538'
     ]

features = [
    'courter_velocity_forward',
    'target_velocity_forward',
    'courter_abs_velocity_lateral',
    'target_abs_velocity_lateral',
    'courter_angles_speed',
    'target_angles_speed',
    'distance_target',
    'relative_angle_abs_target',
    'relative_angle_abs_courter',
    'relative_orientation_abs_target_wrap',
]
condition = 'current_song'

start_frame_idx = 2000
end_frame_idx = -2000
# confidence values not calibrated in SLEAP
confidence_threshold = 0.7 #None if sleap

# filtering
required_sampling_rate = 30#Hz
filter_features = False
filter_cutoff = 10#Hz
filter_ord = 4
assert filter_cutoff < required_sampling_rate/2, (
        "filter_cutoff should be less than %0.2f Hz"
        %(required_sampling_rate/2))
b, a = signal.butter(filter_ord, filter_cutoff/(0.5*required_sampling_rate))

# use normalize_features = True only for wavelet spectrogram approach
normalize_features = True
normalization = "standardize" #"minmax" or None
features_range = None

# time delay works better than wavelet spectrogram for now
time_delay_poses = True
nb_delays = 15
basis_transform = False
basis_function = bases.raised_cosine(0, 12, [0, 12], 10, nb_delays)
wavelet_transform = not time_delay_poses
min_freq = 1
max_freq = 25
nfreqs = 25
freq_spacing = 'dyadic'
freqs, scales_cwt = get_freq_scales(
        min_freq, max_freq, nfreqs, required_sampling_rate,
        spacing = freq_spacing)

# umap parameters
train_size = 0.1
min_dist = 0.0
n_neighbors = 100
random_state = 42
nan_handling = "remove" # "interpolate" to interpolate nans or "remove" to remove nans
if wavelet_transform:
    assert nan_handling == "interpolate", "use nan_handling as interpolate if wavelet_transform is True"

# kde and watershed parameters
bw = 0.5
nb_gridpoints = 256
watershed_threshold = 0.001

results_save_path = f"res/umap_embedding"
if not os.path.exists(results_save_path):
    os.makedirs(results_save_path)

In [ ]:
# save umap parameters
umap_params = {}
umap_params = dict(
    trial_names=trial_names,
    features=features,
    condition=condition,
    required_sampling_rate=required_sampling_rate,
    data_save_path=data_save_path,
    start_frame_idx=start_frame_idx,
    end_frame_idx=end_frame_idx,
    confidence_threshold=confidence_threshold,
    filter_features=filter_features,
    filter_cutoff=filter_cutoff,
    filter_ord=filter_ord,
    normalize_features=normalize_features,
    normalization=normalization,
    features_range=features_range,
    #additional_features=additional_features,
    time_delay_poses=time_delay_poses,
    nb_delays=nb_delays,
    basis_transform = basis_transform,
    basis_function = basis_function,
    wavelet_transform=wavelet_transform,
    min_freq=min_freq,
    max_freq=max_freq,
    nfreqs=nfreqs,
    freq_spacing=freq_spacing,
    train_size=train_size,
    min_dist=min_dist,
    n_neighbors=n_neighbors,
    random_state=random_state,
    nan_handling=nan_handling,
    bw=bw,
    nb_gridpoints=nb_gridpoints,
    watershed_threshold=watershed_threshold,
    results_save_path=results_save_path)

## Prepare the data

Read the data from `preprocessed_data_path` if available. Otherwise, read the trial-wise CSV files in the `dat` folder. Lowpass-filter the data, remove frames below the confidence threshold, and normalize.


In [ ]:
# get data
all_trial_names = list(np.concatenate(list(trial_names.values())))
X, y = [], []
if os.path.exists(preprocessed_data_path):
    data = np.load(preprocessed_data_path, allow_pickle=True)
    X = list(data['X'])
    y = list(data['y'])
else:
    for trial_name in all_trial_names:
        print(trial_name)
        df_trial_data = pd.read_csv(f'{data_save_path}/{trial_name}.csv')
        trial_X = df_trial_data[features].values[start_frame_idx:end_frame_idx]
        trial_y = df_trial_data[condition].values[start_frame_idx:end_frame_idx]
        # low pass filter data
        trial_X = signal.filtfilt(b, a, trial_X, axis=0)
        # filter out non confident frames
        if confidence_threshold is not None and 'poses_confidence' in df_trial_data:
            poses_confidence = df_trial_data['poses_confidence'].values
            confident_frames = poses_confidence>confidence_threshold
            trial_X[~confident_frames] = np.nan
            trial_y[~confident_frames] = np.nan
        # normalization
        if normalize_features:
            if normalization == 'standardize':
                trial_X = StandardScaler().fit_transform(trial_X)
            elif normalization == 'minmax':
                trial_X = MinMaxScaler().fit_transform(trial_X)
        X.append(trial_X)
        y.append(trial_y)
    np.savez(preprocessed_data_path, X=X, y=y)

## Data processing

Extract wavelet spectrograms or perform time-delay embedding of input features for each trial. If using wavelet spectrograms, the processed data for each trial is a 2D array of shape (`N_trial`, `len(features)` x `nfreqs`), where `N_trial` is the number of samples for the trial. If using time-delay embedding, the processed data for each trial is a 2D array of shape (`N_trial`, `len(features)` x `nb_delays`). The processed data from all trials are then pooled to form `X_proc`.



In [ ]:
# process data
X_proc, y_proc = [], []
list_timestamps = []
for trial_X, trial_y in zip(X, y):
    N = trial_X.shape[0]
    t0=0
    dt=1/required_sampling_rate
    timestamps = np.arange(0, N, dtype="float32") * dt + t0
    if wavelet_transform:
        # Compute wavelet spectrogram
        print('Computing wavelet spectrograms ...')
        trial_X_cwt = []
        for feat_idx in range(trial_X.shape[1]):
            [P, t, f] = compute_wavelet(timestamps, trial_X[:, feat_idx],
                                        scales_cwt)
            trial_X_cwt.extend((P))
        trial_X_cwt = np.array(trial_X_cwt, dtype='float32').T
        X_proc.append(trial_X_cwt)
        y_proc.append(trial_y)
        print('done')
    elif time_delay_poses:
        # time delay embedding
        X_td, y_td = time_delay_embedding(
            trial_X, trial_y, nb_delays = nb_delays, multi_features=True,
            padding='same', remove_nans=True if nan_handling == "remove" else False)
        if basis_transform:
            X_td, basis_projection = basis_transformation(
                X_td, nb_delays, basis_function, multi_features=True, nb_stim=len(features))
        X_proc.append(np.array(X_td, dtype="float32"))
        y_proc.append(y_td)
    list_timestamps.append(timestamps)

## UMAP embedding and spatial clustering using KDE and watershed segmentation

First, the time-delayed inputs `X_proc` are embedded into a two-dimensional manifold `X_embedded`. A kernel density estimation is performed on the two-dimensional space, followed by watershed segmentation, which creates spatial clusters centered at local peaks of the kernel density estimates. From a behavioral perspective, the local peaks in the KDE correspond to stereotyped interactions between the organisms, and regions of low density correspond to transitions between stereotyped interactions. Thus, each spatial cluster is assigned to a particular interaction prototype called a social mode.


In [ ]:
if os.path.exists(f'{results_save_path}/embedding_results.npz'):
    # load data
    results = np.load(f'{results_save_path}/embedding_results.npz', allow_pickle=True)
    X_proc = results['X_proc']
    y_proc = results['y_proc']
    X_embedded = results['X_embedded']
    X_embedded_kde = results['X_embedded_kde']
    X_embedded_kde_positions = results['X_embedded_kde_positions']
    X_embedded_segments = results['X_embedded_segments']
    labels = results['labels']
    positions = results['positions']
    labels_edge_positions = results['labels_edge_positions']
else:
    #%% umap embedding
    X_embedded, reducer = umap_embedding(
        X_proc, min_dist, n_neighbors,
        n_components=2,
        train_size=train_size,
        random_state=random_state,
    )
    # kernel density estimation and watershed segmentation
    spatial_clustering = SpatialClustering(bw, nb_gridpoints, watershed_threshold)
    embedded_features_kde, positions, labels, labels_edge, labels_edge_positions = spatial_clustering.fit(X_embedded)
    X_embedded_kde, X_embedded_kde_positions, X_embedded_segments = spatial_clustering.transform(
        X_embedded, positions, labels)

    # save results
    results = {}
    results['X_proc'] = X_proc
    results['y_proc'] = y_proc
    results['timestamps'] = list_timestamps
    results['reducer'] = reducer
    results['X_embedded'] = X_embedded
    results['X_embedded_kde'] = X_embedded_kde
    results['X_embedded_kde_positions'] = X_embedded_kde_positions
    results['X_embedded_segments'] = X_embedded_segments
    results['labels'] = labels
    results['positions'] = positions
    results['labels_edge_positions'] = labels_edge_positions
    results['umap_params'] = umap_params
    np.savez(f'{results_save_path}/embedding_results.npz', results)

## Meaning of each spatial segment (social modes)

To understand what each spatial segment means, we first plot the time-delay embedded features for each social mode separately for male-female and male-male interactions.


In [ ]:
# time-delay embedded feature values for each segment
X_proc_mean = {}
n_samples_label = {}
for analysis_group in trial_names:
    X_proc_mean[analysis_group] = {}
    n_samples_label[analysis_group] = {}
    for label in np.unique(labels):
        if label == 0:
            continue
        X_proc_mean[analysis_group][label] = []
        n_samples_label[analysis_group][label] = 0
        for x_proc, x_seg, trial_id in zip(X_proc, X_embedded_segments, all_trial_names):
            if trial_id not in trial_names[analysis_group]: continue
            X_proc_mean[analysis_group][label].append(
                np.sum(x_proc[x_seg==label], axis=0)/\
                    np.sum(x_seg==label))
            n_samples_label[analysis_group][label] += np.sum(x_seg==label)
        X_proc_mean[analysis_group][label] = np.array(
            X_proc_mean[analysis_group][label])
        if time_delay_poses:
            X_proc_mean[analysis_group][label] = np.reshape(
                X_proc_mean[analysis_group][label],
                (X_proc_mean[analysis_group][label].shape[0], -1, nb_delays))
        elif wavelet_transform:
            X_proc_mean[analysis_group][label] = np.reshape(
                X_proc_mean[analysis_group][label],
                (X_proc_mean[analysis_group][label].shape[0], -1, nfreqs))

num_clusters = 12
fig, ax = plt.subplots(
    len(X_proc_mean),
    num_clusters,
    sharex=True, sharey=True,
    num="cluster_means", figsize=(num_clusters*2, 2*len(X_proc_mean)))
for g, group in enumerate(X_proc_mean):
    for i, (label, features_mean) in enumerate(X_proc_mean[group].items()):
        ax[g, i].set_title(f"{label} ({n_samples_label[group][label]})")
        ax[g, i].matshow(np.nanmean(features_mean, 0), vmin=-1, vmax=1, cmap="bwr")
        ax[g, i].set_yticks(np.arange(features_mean.shape[1]))
        ax[g, i].set_yticklabels(features)
        ax[g, 0].set_ylabel(group)
plt.tight_layout()
plt.show()

Next, we plot the average values of each input feature within each segment. Based on the speed components of each fly and their relative positioning with respect to each other, we name the modes:
1. `Behind and idle`: the courter is behind the target and idle
2. `Behind and close`: the courter is behind the target and close
3. `Behind and chasing`: the courter is behind the target and chasing
4. `Behind and circling`: the courter is behind the target and circling
5. `Uninterested`: the courter and target are distant and facing away from each other
6. `Front and circling`: the courter is in front of the target and circling
7. `Front and close`: the courter is in front of the target and close
8. `Front and idle`: the courter is in front of the target and idle

Some segments contain only a few frames or are noisy, which we ignore.


In [ ]:
# feature means for each segment
filter_by_song = False
ignore_states = [0, 1, 5, 9, 12]
state_names = [
    'Behind idle',
    'Behind close',
    'Behind chasing',
    'Behind circling',
    'Uninterested',
    'Front circling',
    'Front close',
    'Front idle'
]
rows = []
for analysis_group in trial_names:
    for trial_name in trial_names[analysis_group]:
        trial_idx = trial_names[analysis_group].index(trial_name)
        trial_feature_values = StandardScaler().fit_transform(X[trial_idx])
        trial_segments = X_embedded_segments[trial_idx]
        for state in np.unique(trial_segments):
            if state in ignore_states: continue
            for f, feat in enumerate(features):
                trial_feat_segment_mean = np.nanmean(
                    trial_feature_values[trial_segments==state, f])
                rows.append(
                    {"trial_name": trial_name,
                     "analysis_group": analysis_group,
                     "state": state,
                     "feature": feat,
                     "zscore": trial_feat_segment_mean}
                )
df_mean_feature_values_segments = pd.DataFrame(rows)
# plots
figname = 'cluster_feature_means'
fig, ax = plt.subplots(
    1, len(np.unique(df_mean_feature_values_segments.state)),
    figsize=(3*(len(np.unique(df_mean_feature_values_segments.state))),
             len(np.unique(df_mean_feature_values_segments.feature))),
    sharey=True,
    num=figname)
for s, state in enumerate(np.unique(df_mean_feature_values_segments.state)):
    if state in ignore_states: continue
    df_mean_feature_values = df_mean_feature_values_segments[
        df_mean_feature_values_segments.state==state]
    sns.barplot(data=df_mean_feature_values,
                  y="feature",
                  x="zscore",
                  ax=ax[s],
                  dodge=True,
                  #palette=["k"],
                  orient='h')
    #sns.pointplot(data=df_mean_feature_values,
    #            y="feature",
    #            x="zscore",
    #            dodge=0.5,
    #            ax=ax[s],
    #            orient='h',
    #            join=False, palette=['k'])
    ax[s].set_title(f"{state_names[s]}")
    ax[s].set_yticklabels(features)
    ax[s].legend([])
plt.tight_layout()
plt.show()


Next, we visualize the feature values directly on the social UMAP embedding.


In [ ]:
# color code embedding by feature values
binned = True
filter_song = False
groupwise=False
figname = "embedding_feat_values"
if binned: figname = figname + "_binned"
if groupwise: figname = figname + "_groupwise"
if groupwise:
    feat_values_binned = {}
    fig, ax = plt.subplots(
        len(trial_names), len(features), sharex=True, sharey=True,
        num='embedding_feature_values'+("_binned" if binned else "_"),
        figsize=(len(features)*2.5, 6))
    for g, analysis_group in enumerate(trial_names):
        feat_values_binned[analysis_group] = {}
        X_embedded_concat_group = np.concatenate(
            [X_embedded[i] for i in range(len(X_embedded))
             if all_trial_names[i] in trial_names[analysis_group]], axis=0)
        X_raw_concat_group = np.concatenate(
            [X[i] for i in range(len(X))
             if all_trial_names[i] in trial_names[analysis_group]], axis=0)
        X_raw_concat_group = StandardScaler().fit_transform(X_raw_concat_group)
        for i, feature in enumerate(features):
            if binned:
                ret = binned_statistic_2d(
                    X_embedded_concat_group[:, 1],
                    X_embedded_concat_group[:, 0],
                    X_raw_concat_group[:, i],
                    np.nanmean, bins=128,
                    range=[[0, 18], [0, 18]])
                feat_values_binned[analysis_group][feature] = ret
                im = ax[g, i].imshow(
                    ret.statistic,
                    extent=[ret.x_edge[0], ret.x_edge[-1],
                            ret.y_edge[0], ret.y_edge[-1]],
                    cmap='bwr', zorder=1, vmin=-1, vmax=1,)
            else:
                ax[g, i].scatter(
                    *X_embedded_concat_group[::10].T, s=0.1, vmin=-1, vmax=1,
                    alpha=0.25, c=X_raw_concat_group[::10, i], cmap='bwr', zorder=1)
            ax[g, i].scatter(*labels_edge_positions.T, s=0.2, c='k', zorder=2)
            ax[g, i].set_title(feature)
            ax[g, i].set_xlim(0, 18)
            ax[g, i].set_ylim(0, 18)
            ax[g, 0].set_ylabel(analysis_group)
            ax[g, i].axis("off")
        fig.colorbar(
            im, ax=ax[-1, -1], shrink=0.25, aspect=10, ticks=[-1, 0, 1])
# both groups together
else:
    fig, ax = plt.subplots(
        1, len(features), sharex=True, sharey=True,
        num='embedding_feature_values'+("_binned" if binned else "_"),
        figsize=(len(features)*2.5, 3))
    feat_values_binned = {}
    if not filter_song:
        X_embedded_concat_group = np.concatenate(
            [X_embedded[i] for i in range(len(X_embedded))], axis=0)
        X_raw_concat_group = np.concatenate(
            [X[i] for i in range(len(X))], axis=0)
    else:
        X_embedded_concat_group = np.concatenate(
            [X_embedded[i][y[i].ravel()!=0]
             for i in range(len(X_embedded))], axis=0)
        X_raw_concat_group = np.concatenate(
            [X[i][y[i].ravel()!=0] for i in range(len(X))], axis=0)
    X_raw_concat_group = StandardScaler().fit_transform(X_raw_concat_group)
    for i, feature in enumerate(features):
        if binned:
            ret = binned_statistic_2d(
                X_embedded_concat_group[:, 1],
                X_embedded_concat_group[:, 0],
                X_raw_concat_group[:, i],
                np.nanmean, bins=128,
                range=[[-18, 18], [-18, 18]])
            feat_values_binned[feature] = ret
            im = ax[i].imshow(
                ret.statistic,
                extent=[ret.x_edge[0], ret.x_edge[-1],
                        ret.y_edge[0], ret.y_edge[-1]],
                cmap='bwr', zorder=1, vmin=-1, vmax=1,)
        else:
            im = ax[i].scatter(
                *X_embedded_concat_group[::10].T, s=0.1, vmin=-1, vmax=1,
                alpha=0.25, c=X_raw_concat_group[::10, i], cmap='bwr', zorder=1)
        ax[i].scatter(*labels_edge_positions.T, s=0.2, c='k', zorder=2)
        ax[i].set_title(feature, fontsize=10)
        ax[i].set_xlim(0, 20)
        ax[i].set_ylim(0, 20)
        ax[i].axis("off")
    fig.colorbar(im, ax=ax[-1], shrink=0.25, aspect=10, ticks=[-1, 0, 1])
plt.tight_layout()
plt.show()

Finally, we plot the courting male's position around the partner during each social mode. We normalize the distance between flies by the length of the target and limit the visualization to three fly lengths.


In [ ]:
if os.path.exists(f'{results_save_path}/courter_positions.npz'):
    with np.load(f'{results_save_path}/courter_positions.npz', allow_pickle=True) as data:
        dict_courter_positions = data['arr_0'].item()
else:
    dict_courter_positions = {}
    for trial_idx, trial_name in enumerate(all_trial_names):
        df_trial_data = pd.read_csv(f'{data_save_path}/{trial_name}.csv')
        courter_position = df_trial_data[['relative_angle_courter', 'distance_target']].values
        # normalize distance by target length
        courter_position[:, 1] = courter_position[:, 1]/df_trial_data["target_length"].values
        trial_segments = X_embedded_segments[trial_idx]
        for state in np.unique(trial_segments):
            if state in ignore_states: continue
            else:
                if state not in dict_courter_positions:
                    dict_courter_positions[state] = []
                dict_courter_positions[state].extend(courter_position[:len(trial_segments)][trial_segments==state])
    np.savez(f'{results_save_path}/courter_positions.npz', dict_courter_positions)

rbins = np.linspace(0, 3, 31)
abins = np.linspace(-np.pi, np.pi, 73)
courter_position_hist = {}
for state in dict_courter_positions:
    dict_courter_positions[state]=np.array(dict_courter_positions[state])
    azimut = np.array(dict_courter_positions[state][:, 0])
    azimut_rad = np.deg2rad(azimut)
    radius = np.array(dict_courter_positions[state][:, 1])
    #calculate histogram
    hist, _, _ = np.histogram2d(azimut_rad, radius, bins=(abins, rbins),
                                density=True)
    courter_position_hist[state] = hist

# plot
A, R = np.meshgrid(abins, rbins)
figname = 'courter_position_states'
fig, ax = plt.subplots(
    1, len(courter_position_hist), subplot_kw=dict(projection="polar"),
    figsize=(3*len(courter_position_hist), 3), num=figname)
for s, (segment, hist) in enumerate(courter_position_hist.items()):
    pc = ax[s].pcolormesh(
        A, R, (hist).T,
        cmap="Reds",
        vmin=0, vmax=1)
    ax[s].set_title(state_names[s], fontsize=15)
    ax[s].set_xticks([-np.pi*(3/4), -np.pi/2, -np.pi/4,
                      0, np.pi/4, np.pi/2, np.pi*(3/4), np.pi])
    ax[s].set_xlim(-np.pi, np.pi)
    ax[s].set_theta_zero_location("N")
plt.subplots_adjust(bottom=0.4, right=0.9, top=0.6)
cax = plt.axes([0.85, 0.1, 0.01, 0.2])
plt.colorbar(pc, cax=cax)
plt.tight_layout()
plt.show()


## Comparison between male-female and male-male interactions

Once we have the social UMAP embedding, we can compare them across different experimental conditions to understand differences in their interactions. Here we plot the UMAPs for male-female and male-male interactions and visualize their differences.


In [ ]:
# comparison between groups
kde_mean_group = {}
X_embedded_kde_group = {}
groups = list(trial_names.keys())
for group in groups:
    X_embedded_kde_group[group] = []
    for trial_name in trial_names[group]:
        trial_idx = all_trial_names.index(trial_name)
        X_embedded_kde_group[group].append(X_embedded_kde[trial_idx])
    group_kde_mean = np.mean(X_embedded_kde_group[group], 0)
    group_kde_mean[group_kde_mean<1e-3]=0
    kde_mean_group[group] = group_kde_mean

fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(15, 5))
xmin = positions[:,0].min()
xmax = positions[:,0].max()
ymin = positions[:,1].min()
ymax = positions[:,1].max()
for g, group in enumerate(kde_mean_group):
    vmin = 0 #np.round(np.nanpercentile(kde_mean_group[group], 2.5), 2)
    vmax = 0.03 #np.round(np.nanpercentile(kde_mean_group[group], 97.5), 2)
    kde_plot = ax[g].imshow(
        kde_mean_group[group],
        origin='lower',
        extent=[xmin, xmax, ymin, ymax],
        vmin=vmin, vmax=vmax,
        cmap="Reds")
    ax[g].scatter(*labels_edge_positions.T, s=0.2, c='k')
    ax[g].set_title(group)
    ax[g].axis('off')
    ax[g].set_xlim(0, 20)
    ax[g].set_ylim(0, 20)
    fig.colorbar(kde_plot, ax=ax[g], shrink=0.25, aspect=10, ticks=[vmin, vmax])
# difference
statistical_test = True
X_embedded_kde_diff = kde_mean_group[groups[0]] - kde_mean_group[groups[1]]
vmin=-0.03
vmax=0.03
if statistical_test:
    statistic, p_value = mannwhitneyu(X_embedded_kde_group[groups[0]], X_embedded_kde_group[groups[1]])
    X_embedded_kde_diff = X_embedded_kde_diff * (p_value<0.05)
    diff_lim = np.max(np.abs(X_embedded_kde_diff))
kde_diff = ax[2].imshow(
    X_embedded_kde_diff,
    cmap='bwr', origin='lower',
    extent=[xmin, xmax, ymin, ymax],
    vmin=vmin, vmax=vmax)
ax[2].scatter(*labels_edge_positions.T, s=0.2, c='k')
ax[2].set_title("difference")
ax[2].set_xlim(0, 20)
ax[2].set_ylim(0, 20)
ax[2].axis('off')
fig.colorbar(kde_diff, ax=ax[2], shrink=0.25, aspect=10, ticks=[vmin, 0, vmax])
plt.tight_layout()
plt.show()

plt.tight_layout()
plt.show()

Next, we quantify the differences in social modes during male-female and male-male interactions.


In [ ]:
rows = []
for analysis_group in trial_names:
    for trial_name in trial_names[analysis_group]:
        trial_idx = all_trial_names.index(trial_name)
        trial_segments = X_embedded_segments[trial_idx]
        for s, state in enumerate(np.unique(trial_segments)):
            if s in ignore_states: continue
            state_ratio = np.sum(trial_segments==state)/len(trial_segments)
            rows.append(
                {"trial_name": trial_name,
                 "analysis_group": analysis_group,
                 "state": s,
                 "ratio": state_ratio}
            )

df_states_time_spent = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 1, figsize=(2*len(np.unique(df_states_time_spent.state)), 4))
sns.barplot(data=df_states_time_spent, x="state", y="ratio", hue="analysis_group")
sns.stripplot(data=df_states_time_spent, x="state", y="ratio", hue="analysis_group", dodge=True, alpha=0.25, palette=["k"])
sns.despine()
ax.set_ylim(0, 1)
ax.set_xticklabels(state_names)
plt.tight_layout()
plt.show()


## Comparisons between different song contexts

To compare interactions in different singing conditions, we condition our UMAPs on different song contexts and compare song versus silence and pulse versus sine.


In [ ]:
unique_segments = np.unique(np.concatenate(X_embedded_segments))
song_conditioned_kde = {}
rows = []
for analysis_group in trial_names:
    song_conditioned_kde[analysis_group] = {}
    for song_type in ['song', 'silence', 'pulse', 'sine']:
        song_conditioned_kde[analysis_group][song_type] = []
        for datename in trial_names[analysis_group]:
            trial_idx = all_trial_names.index(datename)
            X_embedded_trial = X_embedded[trial_idx][nb_delays:]
            X_embedded_segments_trail = X_embedded_segments[trial_idx][nb_delays:]
            y_trial = y_proc[trial_idx].ravel()
            # song
            if song_type == 'song':
                trial_song_embedding = X_embedded_trial[(y_trial!=0)]
                trial_song_segments = X_embedded_segments_trail[y_trial!=0]
            elif song_type == 'silence':
                trial_song_embedding = X_embedded_trial[(y_trial==0)]
                trial_song_segments = X_embedded_segments_trail[y_trial==0]
            elif song_type == 'pulse':
                trial_song_embedding = X_embedded_trial[(y_trial==1)]
                trial_song_segments = X_embedded_segments_trail[y_trial==1]
            elif song_type == 'sine':
                trial_song_embedding = X_embedded_trial[(y_trial==2)]
                trial_song_segments = X_embedded_segments_trail[y_trial==2]
            _, trial_song_kde = kernel_density_estimation(
                trial_song_embedding, bw, positions)
            song_conditioned_kde[analysis_group][song_type].append(trial_song_kde)
            for state in unique_segments:
                if state not in ignore_states:
                    song_state_time_trial = (
                        np.sum(trial_song_segments==state) / len(trial_song_segments)
                    )
                    rows.append(
                        {"analysis_group": analysis_group,
                         "song": song_type,
                         "trial": datename,
                         "state": state,
                         "time_spent": np.sum(
                             trial_song_segments==state),
                         "time_spent_ratio": song_state_time_trial}
                    )

        song_conditioned_kde[analysis_group][song_type] = np.array(
            song_conditioned_kde[analysis_group][song_type])

df_song_states_time = pd.DataFrame(rows)


In [ ]:
# song vs silence
rows = []
for analysis_group in np.unique(df_song_states_time.analysis_group):
    for trial in np.unique(df_song_states_time.trial):
        for state in np.unique(df_song_states_time.state):
            trial_song_states_time = df_song_states_time[
                (df_song_states_time.analysis_group==analysis_group)&
                (df_song_states_time.trial==trial)&
                (df_song_states_time.state==state)
            ]
            if not len(trial_song_states_time): continue
            song_silence_diff = trial_song_states_time[
                (trial_song_states_time.song=="song")].time_spent_ratio.values[0] - trial_song_states_time[
                (trial_song_states_time.song=="silence")].time_spent_ratio.values[0]
            pulse_sine_diff = trial_song_states_time[
                (trial_song_states_time.song=="pulse")].time_spent_ratio.values[0] - trial_song_states_time[
                (trial_song_states_time.song=="sine")].time_spent_ratio.values[0]
            rows.append(
                            {"analysis_group": analysis_group,
                            "trial": trial,
                            "state": state,
                            "p(song)-p(silence)": song_silence_diff,
                            "p(pulse)-p(sine)": pulse_sine_diff}
            )
df_song_diff = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 3, figsize=(15, 3), gridspec_kw={'width_ratios': [1, 1, 3]})
mean_kde_mf_diff = np.mean(song_conditioned_kde['male-female']['song'], 0) - \
    np.mean(song_conditioned_kde['male-female']['silence'], 0)
mean_kde_mm_diff = np.mean(song_conditioned_kde['male-male']['song'], 0) - \
    np.mean(song_conditioned_kde['male-male']['silence'], 0)
mf_kde = ax[0].matshow(mean_kde_mf_diff,
                    cmap='bwr', origin='lower',
                    extent=[xmin, xmax, ymin, ymax],
                    vmin=-0.03, vmax=0.03)
ax[0].set_xlim(0, 20)
ax[0].set_ylim(0, 20)
ax[0].set_ylabel("song - silence")
ax[0].set_title("female-directed")
ax[0].axis("off")
ax[0].scatter(*labels_edge_positions.T, s=0.2, c='k')
fig.colorbar(mf_kde, ax=ax[0], shrink=0.25, aspect=10, ticks=[-0.03, 0, 0.03])
mm_kde = ax[1].matshow(mean_kde_mm_diff,
                 cmap='bwr', origin='lower',
                extent=[xmin, xmax, ymin, ymax],
                vmin=-0.03, vmax=0.03)
ax[1].set_xlim(0, 20)
ax[1].set_ylim(0, 20)
ax[1].axis("off")
ax[1].set_title("male-directed")
ax[1].scatter(*labels_edge_positions.T, s=0.2, c='k')
#fig.colorbar(mm_kde, ax=ax[1], shrink=0.25, aspect=10, ticks=[-0.03, 0, 0.03])

sns.barplot(data=df_song_diff, x="state", y="p(song)-p(silence)", hue="analysis_group", ax=ax[2])
sns.stripplot(data=df_song_diff, x="state", y="p(song)-p(silence)", hue="analysis_group", ax=ax[2], alpha=0.25, dodge=True, palette=["k"])
ax[2].set_xticklabels(state_names, rotation=45)
plt.suptitle("song - silence")
#plt.tight_layout()


In [ ]:
# pulse vs sine
fig, ax = plt.subplots(1, 3, figsize=(15, 3), gridspec_kw={'width_ratios': [1, 1, 3]})
mean_kde_mf_diff = np.mean(song_conditioned_kde['male-female']['pulse'], 0) - \
    np.mean(song_conditioned_kde['male-female']['sine'], 0)
mean_kde_mm_diff = np.mean(song_conditioned_kde['male-male']['pulse'], 0) - \
    np.mean(song_conditioned_kde['male-male']['sine'], 0)
mf_kde = ax[0].matshow(mean_kde_mf_diff,
                    cmap='bwr', origin='lower',
                    extent=[xmin, xmax, ymin, ymax],
                    vmin=-0.03, vmax=0.03)
ax[0].set_xlim(0, 20)
ax[0].set_ylim(0, 20)
ax[0].set_title("female-directed")
ax[0].axis("off")
ax[0].scatter(*labels_edge_positions.T, s=0.2, c='k')
fig.colorbar(mf_kde, ax=ax[0], shrink=0.25, aspect=10, ticks=[-0.03, 0, 0.03])
mm_kde = ax[1].matshow(mean_kde_mm_diff,
                 cmap='bwr', origin='lower',
                extent=[xmin, xmax, ymin, ymax],
                vmin=-0.03, vmax=0.03)
ax[1].set_xlim(0, 20)
ax[1].set_ylim(0, 20)
ax[1].axis("off")
ax[1].set_title("male-directed")
ax[1].scatter(*labels_edge_positions.T, s=0.2, c='k')
#fig.colorbar(mm_kde, ax=ax[1], shrink=0.25, aspect=10, ticks=[-0.03, 0, 0.03])

sns.barplot(data=df_song_diff, x="state", y="p(pulse)-p(sine)", hue="analysis_group", ax=ax[2])
sns.stripplot(data=df_song_diff, x="state", y="p(pulse)-p(sine)", hue="analysis_group", ax=ax[2], alpha=0.25, dodge=True, palette=["k"])
ax[2].set_xticklabels(state_names, rotation=45)
plt.suptitle("pulse - sine")
#plt.tight_layout()

## Hyperparameter tuning

We used three important hyperparameters: two for fitting the UMAP, `n_neighbors` and `min_dist`, and one for time-delay embedding of features, `nb_delays`.

The `n_neighbors` parameter specifies the size of the local neighborhood UMAP looks at when fitting the manifold. This parameter controls how UMAP balances local versus global structure in the data. Low values make the UMAP fit the local structure well, whereas large values focus on global structure but can lose fine details.

`min_dist` specifies the minimum distance between the embedded points in the low-dimensional manifold. Thus, low values result in points being embedded densely together.

`nb_delays` specifies the history information to be included when embedding the features into low dimensions. Small values focus on immediate history.

To find out the optimal values of these hyperparameters for our data, we performed a hyperparameter optimization. This was done by fitting the UMAP with different hyperparameter combinations and validating performance by the reconstruction error. From the fitted UMAP, an `inverse_transform` is applied on an embedded validation set (not used for fitting), and the mean squared error between the original validation data and reconstructed validation data is quantified.


In [ ]:
#UMAP hyperparameter tuning
n_neighbors_grid=[10, 50, 100, 200]
min_dist_grid=[0.0, 0.1, 0.2, 0.5]
nb_delays_grid=[15, 30, 60, 120]#s
fit_results = umap_multiple_fits(
    X,
    n_neighbors_grid=n_neighbors_grid,
    min_dist_grid=min_dist_grid,
    nb_delays_grid=nb_delays_grid)

# plots
val_score_matrices = {}
for nb_delays in nb_delays_grid:
    val_score_matrices[nb_delays] = np.zeros((len(min_dist_grid), len(n_neighbors_grid)))
    for m, min_dist in enumerate(min_dist_grid):
        for n, n_neighbors in enumerate(n_neighbors_grid):
            val_score_matrices[nb_delays][m, n] = fit_results[
                f'delay: {nb_delays}, '+\
                f'n_components: 2, ' +\
                f'n_neighbors: {n_neighbors}, ' +\
                f'min_dist: {min_dist}, '+\
                f'metric: euclidean'
            ]['val_score']

fig, ax = plt.subplots(1, len(nb_delays_grid), figsize=(len(nb_delays_grid)*3, 3))
for i, nb_delays in enumerate(nb_delays_grid):
    im = ax[i].matshow(val_score_matrices[nb_delays])
    ax[i].set_xlabel("n_neighbors")
    ax[i].set_xticklabels(n_neighbors_grid)
    ax[i].set_ylabel("min_dist")
    ax[i].set_yticklabels(min_dist_grid)
    fig.colorbar(im, ax=ax[i], shrink=0.25, aspect=10)
plt.tight_layout()
plt.show()